# 학습 이미지 검수

232장 학습 이미지를 하나씩 보면서 문제 이미지를 찾는다.
발견한 문제는 아래 `FLAGGED` 리스트에 직접 기록.

체크리스트:
1. 어둡거나 흐림
2. 반사/그림자
3. 알약 경계 잘림
4. 배경 특이
5. 강한 회전
6. 오브젝트 겹침
7. bbox 이상
8. 라벨링 오류 의심
9. 같은 클래스 외형 차이 큼
10. 다른 클래스 외형 유사

## 1. 데이터 다운로드

로컬에 데이터가 이미 있으면 스킵. 없으면 Kaggle secret으로 인증 후 다운로드.

In [ ]:

# kagglehub이 없으면 아래 주석 해제 후 실행
# !uv add kagglehub

import json
from pathlib import Path

# Kaggle 인증 — ~/.kaggle/kaggle.json 생성
try:
    from google.colab import userdata
    kaggle_username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
except ImportError:
    from getpass import getpass
    kaggle_username = input("Kaggle Username: ")
    kaggle_key = getpass("Kaggle API Key: ")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"
kaggle_json_path.write_text(json.dumps({"username": kaggle_username, "key": kaggle_key}))
kaggle_json_path.chmod(0o600)
print("Kaggle 인증 완료")

# data/raw 경로 탐색
current_path = Path.cwd().resolve()

for path_candidate in [current_path, *current_path.parents]:
    raw_root_path = path_candidate / "data" / "raw"
    if raw_root_path.exists():
        break
else:
    raw_root_path = current_path / "data" / "raw"
    raw_root_path.mkdir(parents=True, exist_ok=True)
    print(f"data/raw 폴더 생성: {raw_root_path}")

raw_data_path = raw_root_path / "ai11-level1-project"
dataset_path = raw_data_path / "sprint_ai_project1_data"
required_data_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(d.exists() for d in required_data_dirs):
    print("데이터셋 이미 존재. 다운로드 스킵.")
else:
    import kagglehub
    raw_data_path.mkdir(parents=True, exist_ok=True)
    kagglehub.competition_download("ai11-level1-project", output_dir=str(raw_data_path))

print("데이터셋 경로:", dataset_path)


## 2. 어노테이션 로드

In [ ]:

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

train_image_path = dataset_path / "train_images"
annotation_path = dataset_path / "train_annotations"

annotation_paths = sorted(annotation_path.rglob("*.json"))
bbox_by_image = {}
category_by_id = {}

for annotation_file_path in annotation_paths:
    with annotation_file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for category in data.get("categories", []):
        category_by_id[category["id"]] = category["name"]

    image_id_to_file_name = {img["id"]: img["file_name"] for img in data.get("images", [])}

    for ann in data.get("annotations", []):
        file_name = image_id_to_file_name[ann["image_id"]]
        bbox_by_image.setdefault(file_name, []).append({
            "bbox": ann["bbox"],
            "category_id": ann["category_id"],
            "category_name": category_by_id.get(ann["category_id"], "unknown"),
        })

train_image_files = sorted(train_image_path.glob("*.png"))
print(f"전체 학습 이미지: {len(train_image_files)}장")
print(f"어노테이션 파일: {len(annotation_paths)}개")


## 3. 그리드 보기

한 번에 24장씩 표시. `START_IDX`를 24씩 올려가며 실행.

- `START_IDX = 0` → 이미지 0~23
- `START_IDX = 24` → 이미지 24~47
- `START_IDX = 48` → 이미지 48~71
- ... `START_IDX = 216`까지 (총 10번)

의심 이미지는 제목의 `[N]` 번호를 메모해두고 아래 Detail View에서 자세히 확인.

In [ ]:

START_IDX = 0  # 24씩 올려가며 실행: 0, 24, 48, 72 ...
N_COLS = 4
N_ROWS = 6

batch_files = train_image_files[START_IDX : START_IDX + N_COLS * N_ROWS]
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(20, 30))
axes = axes.flatten()

for ax, image_file in zip(axes, batch_files):
    image = Image.open(image_file).convert("RGB")
    ax.imshow(image)

    for ann in bbox_by_image.get(image_file.name, []):
        x, y, w, h = ann["bbox"]
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

    idx = train_image_files.index(image_file)
    n_bbox = len(bbox_by_image.get(image_file.name, []))
    ax.set_title(f"[{idx}] bbox:{n_bbox}\n{image_file.name[:28]}", fontsize=7)
    ax.axis("off")

for ax in axes[len(batch_files):]:
    ax.axis("off")

plt.suptitle(f"Images [{START_IDX} - {START_IDX + len(batch_files) - 1}]", fontsize=14)
plt.tight_layout()
plt.show()


## 4. 이미지 자세히 보기

`TARGET_IDX`에 그리드에서 메모한 번호 입력.

In [ ]:

TARGET_IDX = 0  # 그리드에서 메모한 번호 입력

target_file = train_image_files[TARGET_IDX]
image = Image.open(target_file).convert("RGB")
bboxes = bbox_by_image.get(target_file.name, [])

fig, ax = plt.subplots(figsize=(10, 13))
ax.imshow(image)

colors = ["lime", "red", "cyan", "yellow"]
for i, ann in enumerate(bboxes):
    x, y, w, h = ann["bbox"]
    color = colors[i % len(colors)]
    rect = patches.Rectangle((x, y), w, h, linewidth=3, edgecolor=color, facecolor="none")
    ax.add_patch(rect)
    ax.text(
        x, max(y - 5, 0),
        ann["category_name"],
        color=color,
        fontsize=9,
        bbox={"facecolor": "black", "alpha": 0.6, "pad": 2},
    )

ax.set_title(f"[{TARGET_IDX}] {target_file.name}", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"파일명: {target_file.name}")
print(f"bbox 수: {len(bboxes)}")
for i, ann in enumerate(bboxes):
    print(f"  [{i}] {ann['category_name']} — bbox: {ann['bbox']}")


## 5. 문제 이미지 기록

`FLAGGED` 리스트에 직접 추가. 검수 완료 후 옵시디언 `experiment/이미지 검수.md`에 옮겨 적기.

In [ ]:

CHECKLIST = [
    "어둡거나 흐림",
    "반사/그림자",
    "알약 경계 잘림",
    "배경 특이",
    "강한 회전",
    "오브젝트 겹침",
    "bbox 이상",
    "라벨링 오류 의심",
    "같은 클래스 외형 차이 큼",
    "다른 클래스 외형 유사",
]

FLAGGED = [
    # 형식: {"idx": 이미지번호, "file": "파일명.png", "issues": [항목번호, ...], "memo": "메모"}
    # 예시:
    # {"idx": 5, "file": "K-xxx.png", "issues": [1, 3], "memo": "매우 어두움, 오른쪽 알약 잘림"},
]

print(f"기록된 문제 이미지: {len(FLAGGED)}건")
for item in FLAGGED:
    issues_str = ", ".join([CHECKLIST[i - 1] for i in item["issues"]])
    print(f"  [{item['idx']}] {item['file']}")
    print(f"       문제: {issues_str}")
    print(f"       메모: {item['memo']}")
